# Analyses for Relational Network Data

This notebook performs robustness analyses on the rural resilience survey data.

## Workflow:
1. Load survey data from TSV files (participants, questions, answers)
2. Perform robustness analyses by dropping participant groups
3. Compare network metrics across different participant subsets

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
from typing import List, Dict, Tuple
import networkx as nx
import sys

# Add parent directory to path for importing local modules
sys.path.insert(0, str(Path('../nbra').resolve()))

In [ ]:
# Import network analysis utilities
from nbra import network_analysis as na

In [ ]:
# Define paths
working_country = 'tr'
data_dir = Path('../data/02_relational_network', working_country)
output_dir = Path('../results/02_relational_network', working_country)
output_dir.mkdir(parents=True, exist_ok=True)


if working_country == 'de':
    nodes_file = data_dir / 'de_nodes.csv'
    edges_file = data_dir / 'de_edges.csv'
else:
    nodes_file = data_dir / 'tr_nodes.csv'
    edges_file = data_dir / 'tr_edges.csv'

In [ ]:
# Load node data for matching
df_nodes = pd.read_csv(nodes_file)
print(f"\nNode data shape: {df_nodes.shape}")
print(f"\nNode columns: {df_nodes.columns.tolist()}")
df_nodes.head()

## Network Analysis - Calculate Baseline Network Metrics

This section builds the relational network using the edges file and calculates weighted degree for all nodes.

In [ ]:
# Load edges data
df_edges = pd.read_csv(edges_file)
print(f"Edges data shape: {df_edges.shape}")
print(f"\nEdges columns: {df_edges.columns.tolist()}")
print(f"\nFirst few edges:")
df_edges.head(10)

# Load df_answers, df_questions, and df_participants from previously saved TSV files
answers_file = data_dir / 'answers.tsv'
questions_file = data_dir / 'questions.tsv'
participants_file = data_dir / 'participants.tsv'

df_answers = pd.read_csv(answers_file, sep='\t')
df_questions = pd.read_csv(questions_file, sep='\t')
df_participants = pd.read_csv(participants_file, sep='\t')

print(f"\n\nLoaded df_answers from file: {answers_file}")
print(f"Answers data shape: {df_answers.shape}")
print(f"Answers columns: {df_answers.columns.tolist()}")
print(f"\nFirst few answers:")
print(df_answers.head(10))

print(f"\n\nLoaded df_questions from file: {questions_file}")
print(f"Questions data shape: {df_questions.shape}")
print(f"Questions columns: {df_questions.columns.tolist()}")
print(f"\nFirst few questions:")
print(df_questions.head(10))

print(f"\n\nLoaded df_participants from file: {participants_file}")
print(f"Participants data shape: {df_participants.shape}")
print(f"Participants columns: {df_participants.columns.tolist()}")
print(f"\nFirst few participants:")
print(df_participants.head(10))

In [ ]:
# Build network and calculate metrics
# This function: 
#   1. Calculates avg_scores from answers
#   2. Matches edges to questions
#   3. Recalculates edge weights as (source_avg + target_avg) / 2
#   4. Builds NetworkX graph
#   5. Calculates weighted degree for all nodes
df_questions_enriched = na.build_network_and_calculate_metrics(df_answers, df_questions, df_edges)

print("Questions with network metrics:")
print(df_questions_enriched[['questionId', 'node_label', 'node_type', 'avg_score', 'weighted_degree']].head(20))

In [ ]:
# Rank Solutions by weighted degree
df_solutions_ranked = na.rank_solutions(df_questions_enriched)

print(f"\nSolutions ranked by weighted degree:")
print(f"Total Solutions: {len(df_solutions_ranked)}\n")
print(df_solutions_ranked[['rank', 'node_label', 'avg_score', 'weighted_degree']].to_string(index=False))

In [ ]:
# Save updated questions with network metrics
questions_enriched_file = output_dir / 'questions_with_metrics.tsv'
df_questions_enriched.to_csv(questions_enriched_file, sep='\t', index=False)
print(f"Saved enriched questions data to: {questions_enriched_file}")

# Save ranked solutions
solutions_ranked_file = output_dir / 'solutions_ranked.tsv'
df_solutions_ranked.to_csv(solutions_ranked_file, sep='\t', index=False)
print(f"Saved ranked solutions to: {solutions_ranked_file}")

## Perturbation Analysis Framework (Jackknife / Bootstrap)

This section provides a template for running robustness analyses through data perturbations.
The modular functions allow you to:
1. Perturb the df_answers dataframe (e.g., drop participants, resample)
2. Recalculate network metrics using the same functions
3. Collect rankings across iterations
4. Compare rank stability

In [ ]:
# Example: Jackknife analysis - drop one participant at a time

jackknife_rankings = []

for participant_id in df_participants['participantId']:
    # Create perturbed dataset: exclude one participant
    df_answers_perturbed = df_answers[df_answers['participantId'] != participant_id]
    
    # Recalculate metrics with perturbed data
    df_questions_perturbed = na.build_network_and_calculate_metrics(
        df_answers_perturbed, df_questions, df_edges
    )
    
    # Get solution rankings
    df_solutions_perturbed = na.rank_solutions(df_questions_perturbed)
    
    # Store rankings
    jackknife_rankings.append({
        'excluded_participant': participant_id,
        'rankings': df_solutions_perturbed[['node_label', 'rank', 'weighted_degree']].copy()
    })

print(f"Completed {len(jackknife_rankings)} jackknife iterations")

In [ ]:
## Robustness Analysis: Jackknife with Multiple Parameters

print("="*80)
print("ROBUSTNESS ANALYSIS - JACKKNIFE")
print("="*80)

# Define parameter ranges to test
exclusion_counts = [1,2,3,4]
score_cutoffs = [2,2.5,3]

# Store all results for final TSV output
all_results = []

# Loop over all parameter combinations
for exclusion_count in exclusion_counts:
    for score_cutoff in score_cutoffs:
        print(f"\n{'='*80}")
        print(f"Testing: exclusion_count={exclusion_count}, score_cutoff={score_cutoff}")
        print(f"{'='*80}")

        # Generate exclusion combinations for this exclusion_count
        jackknife_exclusions = na.generate_jackknife_exclusions(
            df_participants['participantId'].tolist(),
            exclusion_count
        )
        print(f"Total jackknife iterations: {len(jackknife_exclusions)}")

        # Calculate baseline for this score_cutoff
        df_questions_baseline = na.build_network_and_calculate_metrics(
            df_answers, df_questions, df_edges, None
        )
        df_solutions_baseline = na.rank_solutions(df_questions_baseline)

        # Run jackknife analysis
        jackknife_comparisons = []

        for excluded_participants in jackknife_exclusions:
            # Create perturbed dataset: exclude selected participants
            df_answers_perturbed = df_answers[
                ~df_answers['participantId'].isin(excluded_participants)
            ]

            # Recalculate metrics with perturbed data
            df_questions_perturbed = na.build_network_and_calculate_metrics(
                df_answers_perturbed, df_questions, df_edges, score_cutoff
            )

            # Get solution rankings
            df_solutions_perturbed = na.rank_solutions(df_questions_perturbed)

            # Compare with baseline for this score_cutoff
            comparison = na.compare_rankings(
                baseline_df=df_solutions_baseline,
                perturbed_df=df_solutions_perturbed,
                top_n_values=[3, 5, 7, 10],
                id_column='questionId',
                rank_column='rank'
            )

            jackknife_comparisons.append(comparison)

            # Progress indicator (every 100 iterations or at end)
            iter_count = len(jackknife_comparisons)
            if iter_count % 500 == 0 or iter_count == len(jackknife_exclusions):
                print(f"  Progress: {iter_count}/{len(jackknife_exclusions)} iterations completed")

        # Calculate summary statistics for this parameter combination
        jackknife_summary = pd.DataFrame(jackknife_comparisons)
        jackknife_means = jackknife_summary.mean()
        print(f"Jackknife means:")
        print(jackknife_summary.describe())

        # Store results
        result_row = {
            'Cut-off': score_cutoff,
            'Exclude': exclusion_count,
            'jaccard_top_3': jackknife_means['jaccard_top_3'],
            'jaccard_top_5': jackknife_means['jaccard_top_5'],
            'jaccard_top_7': jackknife_means['jaccard_top_7'],
            'jaccard_top_10': jackknife_means['jaccard_top_10'],
            'Spearman': jackknife_means['spearman']
        }
        all_results.append(result_row)

        print(f"\nResults for exclusion_count={exclusion_count}, score_cutoff={score_cutoff}:")
        print(f"  Jaccard Top-3: {result_row['jaccard_top_3']:.2f}")
        print(f"  Jaccard Top-5: {result_row['jaccard_top_5']:.2f}")
        print(f"  Jaccard Top-7: {result_row['jaccard_top_7']:.2f}")
        print(f"  Jaccard Top-10: {result_row['jaccard_top_10']:.2f}")
        print(f"  Spearman: {result_row['Spearman']:.2f}")

# Create final results DataFrame
df_robustness_results = pd.DataFrame(all_results)

print(f"\n\n{'='*80}")
print("FINAL RESULTS TABLE")
print(f"{'='*80}\n")
print(df_robustness_results.to_string(index=False))

# Save results to TSV file
robustness_results_file = output_dir / 'jackknife_parameter_sweep_results.tsv'
df_robustness_results.to_csv(robustness_results_file, sep='\t', index=False)
print(f"\n\nSaved robustness results to: {robustness_results_file}")

In [ ]:
# ============================================================================
# BOOTSTRAP ANALYSIS (n=1000)
# ============================================================================
print("\n\n2. BOOTSTRAP ANALYSIS (Resample with replacement)")
print("-"*80)

n_bootstrap = 1000
print(f"Total bootstrap iterations: {n_bootstrap}")

# Set random seed for reproducibility
np.random.seed(42)

bootstrap_comparisons = []

for iteration in range(n_bootstrap):
    # Resample participants with replacement
    resampled_participant_ids = np.random.choice(
        df_participants['participantId'], 
        size=len(df_participants), 
        replace=True
    )
    
    # Create perturbed dataset with properly duplicated participants
    # Need to handle participants that appear multiple times
    df_answers_list = []
    for pid in resampled_participant_ids:
        participant_answers = df_answers[df_answers['participantId'] == pid].copy()
        df_answers_list.append(participant_answers)
    
    df_answers_perturbed = pd.concat(df_answers_list, ignore_index=True)

    # print(f"bootstrap vs: {len(df_answers)} vs {len(df_answers_perturbed)}")

    # Recalculate metrics
    df_questions_perturbed = na.build_network_and_calculate_metrics(
        df_answers_perturbed, df_questions, df_edges
    )
    
    # Get solution rankings
    df_solutions_perturbed = na.rank_solutions(df_questions_perturbed)
    
    
    # Compare with baseline
    comparison = na.compare_rankings(
        baseline_df=df_solutions_ranked,
        perturbed_df=df_solutions_perturbed,
        top_n_values=[3, 5, 7, 10],
        id_column='questionId',
        rank_column='rank'
    )
    
    bootstrap_comparisons.append(comparison)
    
    # Progress indicator
    if (iteration + 1) % 250 == 0:
        print(f"  Progress: {iteration + 1}/{n_bootstrap} iterations completed")

print(f"\nCompleted {len(bootstrap_comparisons)} bootstrap iterations")

# Calculate summary statistics for bootstrap
bootstrap_summary = pd.DataFrame(bootstrap_comparisons)
bootstrap_means = bootstrap_summary.mean()

print("\nBootstrap Summary Statistics:")
print(bootstrap_summary.describe())

In [ ]:
import numpy as np
import pandas as pd

print("\n\n2. BOOTSTRAP ANALYSIS (Resample with replacement)")
print("-" * 80)
score_cutoffs = [2,2.5,3]

# Store all results for final TSV output
all_results = []


for score_cutoff in score_cutoffs:
    # --- config ---
    n_bootstrap = 5000  # set to 5000 for your final run
    print(f"Total bootstrap iterations: {n_bootstrap}")
    DEBUG_SANITY = True   # set False after you validate once

    # --- baseline objects assumed available ---
    # df_participants, df_answers, df_questions, df_edges
    # df_solutions_ranked (baseline ranking, with columns: 'questionId', 'rank')

    # --- prepare population: unique participant IDs ---
    participant_ids = pd.Index(df_participants['participantId'].dropna().unique())
    n = participant_ids.size
    assert n > 0, "No participants found."
    print(f"Unique participants: n = {n}")

    # Optional: guardrail if your expectation is 20
    # assert n == 20, f"Expected 20 participants, got {n}"

    # --- single RNG for the entire process (reproducible + varying draws) ---
    rng = np.random.default_rng(42)

    bootstrap_comparisons = []
    unique_counts_tracker = []   # number of unique respondents in each replicate
    mult_hist_tracker = []       # store counts histogram (0,1,2,3,...) for first few reps

    # pre-allocate baseline answers for merge (speed)
    df_answers_base = df_answers.copy()

    # progress reporting cadence
    report_every = max(1, n_bootstrap // 10)

    for iteration in range(n_bootstrap):
        # --- 1) sample multiplicities: bootstrap with replacement over participants ---
        # Equivalent to drawing 'n' ids with replacement, but explicit about per-ID counts.
        mult = pd.Series(
            rng.multinomial(n, np.full(n, 1.0 / n, dtype=float)),
            index=participant_ids,
            name="rep_weight",
        )  # integers summing to n

        # --- 2) build perturbed dataset by repeating rows according to multiplicity ---
        # If you can modify your network builder to accept weights, you can avoid row repetition
        # and pass 'rep_weight' as a weight column (faster). For now, we repeat rows explicitly.
        df_answers_perturbed = (
            df_answers_base.merge(mult, left_on="participantId", right_index=True, how="left")
                        .fillna({"rep_weight": 0})
        )
        df_answers_perturbed["rep_weight"] = df_answers_perturbed["rep_weight"].astype("int32")
        df_answers_perturbed = df_answers_perturbed[df_answers_perturbed["rep_weight"] > 0]
        df_answers_perturbed = df_answers_perturbed.loc[
            df_answers_perturbed.index.repeat(df_answers_perturbed["rep_weight"])
        ].drop(columns="rep_weight").reset_index(drop=True)

        # --- 3) recompute metrics on the perturbed dataset ---
        df_questions_perturbed = na.build_network_and_calculate_metrics(
            df_answers_perturbed, df_questions, df_edges, score_cutoff
        )
        df_solutions_perturbed = na.rank_solutions(df_questions_perturbed)

        # --- 4) compare to baseline ---
        comparison = na.compare_rankings(
            baseline_df=df_solutions_ranked,
            perturbed_df=df_solutions_perturbed,
            top_n_values=[3, 5, 7, 10],
            id_column='questionId',
            rank_column='rank'
        )
        bootstrap_comparisons.append(comparison)

        # --- optional sanity tracking ---
        unique_counts_tracker.append(int((mult > 0).sum()))
        if DEBUG_SANITY and iteration < 3:
            # show distribution of multiplicities for first few reps
            hist = mult.value_counts().sort_index()  # counts of 0,1,2,3,... occurrences
            mult_hist_tracker.append(hist)

        # progress
        if (iteration + 1) % report_every == 0 or (iteration + 1) == n_bootstrap:
            print(f"  Progress: {iteration + 1}/{n_bootstrap} iterations completed")

    print(f"\nCompleted {len(bootstrap_comparisons)} bootstrap iterations")

    # --- 5) summarize bootstrap comparisons ---
    bootstrap_summary = pd.DataFrame(bootstrap_comparisons)
    print("\nBootstrap Summary Statistics:")
    print(bootstrap_summary.describe())  # mean, std, min, max, quartiles

    # You can also pull the means/medians explicitly if needed:
    bootstrap_means = bootstrap_summary.mean(numeric_only=True)
    bootstrap_medians = bootstrap_summary.median(numeric_only=True)

    print("\nBootstrap Means:\n", bootstrap_means.to_string())
    print("\nBootstrap Medians:\n", bootstrap_medians.to_string())

    # Store results
    result_row = {
        'Cut-off': score_cutoff,
        'jaccard_top_3': bootstrap_means['jaccard_top_3'],
        'jaccard_top_5': bootstrap_means['jaccard_top_5'],
        'jaccard_top_7': bootstrap_means['jaccard_top_7'],
        'jaccard_top_10': bootstrap_means['jaccard_top_10'],
        'Spearman': bootstrap_means['spearman']
    }
    all_results.append(result_row)

    print(f"\nResults for score_cutoff={score_cutoff}:")
    print(f"  Jaccard Top-3: {result_row['jaccard_top_3']:.2f}")
    print(f"  Jaccard Top-5: {result_row['jaccard_top_5']:.2f}")
    print(f"  Jaccard Top-7: {result_row['jaccard_top_7']:.2f}")
    print(f"  Jaccard Top-10: {result_row['jaccard_top_10']:.2f}")
    print(f"  Spearman: {result_row['Spearman']:.2f}")

    # --- 6) sanity checks (expected patterns for true bootstrap) ---
    if DEBUG_SANITY:
        # (a) average number of unique respondents per replicate ~ 12.6 when n=20
        #     (in general: n * (1 - (1 - 1/n)**n))
        avg_unique = float(np.mean(unique_counts_tracker))
        print(f"\n[Sanity] Avg unique respondents per replicate: {avg_unique:.2f}")

        # (b) early multiplicity histograms (should show many 0s & 1s, some 2s, a few 3s)
        for i, h in enumerate(mult_hist_tracker):
            print(f"[Sanity] multiplicity histogram (replicate {i}):")
            # ensure we show 0..3 even if some bins are missing
            for k in range(0, max(4, int(h.index.max() if len(h) else 0) + 1)):
                print(f"  count=={k}: {int(h.get(k, 0))}")

# Create final results DataFrame
df_robustness_results = pd.DataFrame(all_results)

print(f"\n\n{'='*80}")
print("FINAL RESULTS TABLE")
print(f"{'='*80}\n")
print(df_robustness_results.to_string(index=False))

# Save results to TSV file
robustness_results_file = output_dir / 'bootstrap_parameter_sweep_results.tsv'
df_robustness_results.to_csv(robustness_results_file, sep='\t', index=False)
print(f"\n\nSaved robustness results to: {robustness_results_file}")

In [ ]:
# ============================================================================
# SUMMARY TABLE
# ============================================================================
print("\n\n" + "="*80)
print("FINAL SUMMARY TABLE")
print("="*80)

summary_data = {
    'Method': [
        f'Jackknife (n={len(jackknife_comparisons)})',
        f'Bootstrap (n={n_bootstrap})'
    ],
    'Jaccard_Top3': [
        jackknife_means['jaccard_top_3'],
        bootstrap_means['jaccard_top_3']
    ],
    'Jaccard_Top5': [
        jackknife_means['jaccard_top_5'],
        bootstrap_means['jaccard_top_5']
    ],
    'Jaccard_Top7': [
        jackknife_means['jaccard_top_7'],
        bootstrap_means['jaccard_top_7']
    ],
    'Jaccard_Top10': [
        jackknife_means['jaccard_top_10'],
        bootstrap_means['jaccard_top_10']
    ],
    'Spearman_Correlation': [
        jackknife_means['spearman'],
        bootstrap_means['spearman']
    ]
}

df_robustness_summary = pd.DataFrame(summary_data)
print("\n" + df_robustness_summary.to_string(index=False))

# Save results
robustness_summary_file = output_dir / 'robustness_summary.tsv'
df_robustness_summary.to_csv(robustness_summary_file, sep='\t', index=False)
print(f"\n\nSaved robustness summary to: {robustness_summary_file}")

# Save detailed results
jackknife_details_file = output_dir / 'jackknife_detailed_results.tsv'
jackknife_summary.to_csv(jackknife_details_file, sep='\t', index=False)
print(f"Saved detailed jackknife results to: {jackknife_details_file}")

bootstrap_details_file = output_dir / 'bootstrap_detailed_results.tsv'
bootstrap_summary.to_csv(bootstrap_details_file, sep='\t', index=False)
print(f"Saved detailed bootstrap results to: {bootstrap_details_file}")